# SLIIT IT3051 - EV Battery PHM Dual-Task System
## Week 2 - Task 1: Non-Linear Tree & Ensemble Regressors (RUL Prediction)
### **Assigned Member: Person B**

---
### Overview & Objectives:
Implement non-linear Decision Trees, Random Forest Regressors, Gradient Boosting ensembles, feature importance attribution, and residual error diagnostics.


### 0. Environment Setup & Data Partition Loading
Execute this cell to load the verified preprocessed matrices.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Visual aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)

def load_data():
    dataset_path = 'ev battery_failure  Dataset.csv'
    for p in [dataset_path, f'../{dataset_path}', f'../../{dataset_path}']:
        if os.path.exists(p):
            dataset_path = p
            break
            
    df = pd.read_csv(dataset_path)
    
    # 7 Domain Features
    df['temperature_spread'] = df['cell_temperature_max'] - df['cell_temperature_avg']
    df['efficiency_gap'] = (df['charge_efficiency'] - df['discharge_efficiency']).abs()
    df['health_loss_interaction'] = (df['battery_health_percent'] * df['capacity_loss_percent']) / 100.0
    df['resistance_per_1000_cycles'] = (df['internal_resistance'] / (df['cycle_count'] + 1.0)) * 1000.0
    df['c_rate_proxy'] = df['average_charge_power_kw'] / (df['battery_capacity_kwh'] + 1e-5)
    df['cell_voltage_spread'] = df['cell_voltage_std'] / (df['cell_voltage_avg'] + 1e-5)
    df['stress_index'] = df['aggressive_acceleration_score'] * df['hard_braking_score']
    
    id_cols = ['vehicle_id', 'battery_serial']
    target_rul = 'predicted_remaining_life_cycles'
    target_fail = 'battery_failure'
    
    excluded = id_cols + [target_rul, target_fail]
    features = [c for c in df.columns if c not in excluded]
    
    num_cols = df[features].select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    
    num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    
    # Task 1 Splits (RUL)
    df_t1 = df[df[target_rul].notnull()].copy()
    X1_tr, X1_te, y1_train, y1_test = train_test_split(df_t1[features], df_t1[target_rul], test_size=0.20, random_state=42)
    ct1 = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)], verbose_feature_names_out=False)
    X1_train = ct1.fit_transform(X1_tr)
    X1_test = ct1.transform(X1_te)
    
    # Task 2 Splits (Failure)
    X2_tr, X2_te, y2_train, y2_test = train_test_split(df[features], df[target_fail], test_size=0.20, random_state=42, stratify=df[target_fail])
    ct2 = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)], verbose_feature_names_out=False)
    X2_train = ct2.fit_transform(X2_tr)
    X2_test = ct2.transform(X2_te)
    
    feature_names = ct1.get_feature_names_out()
    return (X1_train, X1_test, y1_train, y1_test), (X2_train, X2_test, y2_train, y2_test), feature_names

print("Data Loader initialized successfully!")



In [ ]:
(X1_train, X1_test, y1_train, y1_test), _, feature_names = load_data()
print(f'Task 1 Train Shape: {X1_train.shape} | Test Shape: {X1_test.shape}')


### Task 1: Non-Linear Baseline: Decision Tree Regressor
Train a single Decision Tree Regressor. Test varying max_depth values (e.g., 3, 5, 10, None) to demonstrate tree overfitting.


In [ ]:
# TODO: Import DecisionTreeRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# TODO: Train DecisionTreeRegressor with default parameters vs constrained max_depth

# TODO: Compare train vs test R^2 to detect overfitting



**Analysis & Viva Preparation Notes (Fill this in after running code):**
- What happens to the training R^2 vs test R^2 when max_depth is unconstrained?
- Why do single decision trees struggle with continuous regression extrapolation?


### Task 2: Bagging Ensemble: Random Forest Regressor
Train a Random Forest Regressor (n_estimators=100). Evaluate test R^2, RMSE, and MAE. Extract and plot the Top 15 Feature Importances.


In [ ]:
# TODO: Import RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor

# TODO: Train RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# TODO: Evaluate performance metrics and extract feature_importances_

# TODO: Plot horizontal bar chart of Top 15 most important features



**Analysis & Viva Preparation Notes (Fill this in after running code):**
- How much does Random Forest improve over a single Decision Tree and Linear Regression?
- Which top 3 physical features contribute the most variance reduction according to the tree splits?


### Task 3: Boosting Ensemble: Gradient Boosting / XGBoost Regressor
Train a Gradient Boosting model (HistGradientBoostingRegressor or XGBRegressor). Perform 5-fold cross-validation and tune max_depth and learning_rate.


In [ ]:
# TODO: Import HistGradientBoostingRegressor or XGBRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

# TODO: Train boosting model with cross-validation

# TODO: Evaluate performance metrics on test partition



**Analysis & Viva Preparation Notes (Fill this in after running code):**
- Did Gradient Boosting outperform Random Forest? Explain why sequential error-correction works well here.
- What learning_rate and depth yielded the best validation score?


### Task 4: Residual Error Diagnostics & Homoscedasticity Analysis
Plot residual diagnostics: (1) Residual scatter plot (y_test vs y_test - y_pred), and (2) Residual error distribution histogram.


In [ ]:
# TODO: Calculate residuals = y1_test - y_pred_best

# TODO: Plot residual scatter plot and residual histogram



**Analysis & Viva Preparation Notes (Fill this in after running code):**
- Are residuals centered around zero, and are they homoscedastic (constant variance across all cycle ranges)?
- Does the model make larger prediction errors for highly aged batteries (< 3,000 cycles)?
